
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Multiplex Streaming SDP with Delta Sinks and Iceberg Reads

#### Duration: ~31-36 Minutes

## Overview

This demonstration showcases how to build a multiplex data pipeline using Spark Declarative Pipelines (Lakeflow). You will learn how to ingest multiplexed data from a single source into a bronze streaming table, then demultiplex that data into multiple silver streaming tables based on event types, and finally create aggregated gold-layer views for analytics.

The demo simulates a real-world scenario where multiple business domains (store operations, marketing, and logistics) generate events that are ingested into a single data stream. Using the medallion architecture pattern, you will process this multiplexed data through bronze, silver, and gold layers while enabling Iceberg compatibility for cross-platform analytics.

## Learning Objectives

By the end of this demonstration, you will be able to:

- **Build multiplex data pipelines** using Spark Declarative Pipelines to handle multiple event types from a single source
- **Demultiplex streaming data** by filtering and transforming events into separate domain-specific tables
- **Build Delta sinks** using the Python API to create analytics-ready tables from streaming sources
- **Enable Iceberg compatibility** on Delta tables to support cross-platform data access and analytics

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 6**  
  - [How to select an environment version](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)
  </div>
</div>


### Multiplex Pipeline Demonstration Overview
In this demonstration, you'll build a Apache Spark™ Declarative Pipeline that implements the full medallion architecture, from raw data ingestion to curated analytics.

1. **Ingest multiplexed files** from cloud storage and write them into a single bronze table.  
    - These files contain multiplexed data, meaning they include events for **marketing**, **logistics**, and **store operations**, all dumped into a single cloud location.
2. **Demultiplex the bronze table** into intermediate tables based on event group. Three intermediate tables will be created in the bronze layer.
3. **Transform intermediate tables** to create silver tables for each event group, adding new columns as required by business logic and casting columns to the appropriate data types.
4. **Create gold materialized views** from the marketing data that automatically refresh and provide analytics-ready results.
5. **Write the logistics table to a Delta sink** to enable external access via Iceberg reads.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1300 610" width="100%" role="img" aria-label="Multiplex pipeline overview: one business-events source feeds a bronze table that is de-multiplexed into intermediate, silver, then a Delta sink and a gold materialized view" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif;max-width:1300px;display:block;margin:12px auto;"><defs><marker id="om-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#7C8B94"/></marker><marker id="om-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="om-ah-t" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B5062"/></marker><filter id="om-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><text x="14" y="30" font-size="20" font-weight="400" fill="#1B3139" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Cloud Storage</text><rect x="10" y="42" width="150" height="330" rx="18" fill="#fff" stroke="#9AA5AB" stroke-width="1.6" /><rect x="22" y="177" width="126" height="150" rx="8" fill="#fff" stroke="#8C98A0" stroke-width="1.5" stroke-dasharray="5 4" /><text x="85" y="201" font-size="12.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Business Events</text><text x="85" y="218" font-size="12.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Source</text><g transform="translate(33,247) scale(0.78)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >PQ</text></g><g transform="translate(71,247) scale(0.78)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >PQ</text></g><g transform="translate(109,247) scale(0.78)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >PQ</text></g><text x="85" y="410" font-size="14" font-weight="400" fill="#1B3139" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Files are landed into</text><text x="85" y="429" font-size="14" font-weight="700" fill="#1B3139" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >volume daily</text><rect x="196" y="180" width="200" height="200" rx="2" fill="none" stroke="#FFAB00" stroke-width="1.6" stroke-dasharray="6 5" /><svg x="208" y="188" width="30" height="27.3" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="246" y="208" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="208" y="230" width="176" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="296.0" y="257" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >bronze_demo</text><rect x="204" y="292" width="184" height="80" rx="0" fill="#F6F4F1" stroke="none" stroke-width="1" /><text x="296" y="322" font-size="14.5" text-anchor="middle" fill="#1B3139" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif">Ingesting <tspan font-weight="700">files</tspan> from</text><text x="296" y="344" font-size="14.5" text-anchor="middle" fill="#1B3139" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif"><tspan font-weight="700">source</tspan></text><path d="M162,252 H194" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><rect x="440" y="30" width="262" height="470" rx="2" fill="none" stroke="#FFAB00" stroke-width="1.6" stroke-dasharray="6 5" /><svg x="456" y="58" width="30" height="27.3" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="494" y="78" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="456" y="100" width="230" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="571.0" y="127" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >logistics_intermediate</text><path d="M398,252 H410 V130 Q410,122 418,122 H451" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="456" y="188" width="30" height="27.3" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="494" y="208" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="456" y="230" width="230" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="571.0" y="257" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >marketing_intermediate</text><path d="M398,252 H451" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="456" y="318" width="30" height="27.3" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="494" y="338" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="456" y="360" width="230" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="571.0" y="387" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >store_ops_intermediate</text><path d="M398,252 H410 V374 Q410,382 418,382 H451" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><rect x="452" y="440" width="238" height="50" rx="0" fill="#F6F4F1" stroke="none" stroke-width="1" /><text x="571" y="462" font-size="13.5" font-weight="400" fill="#1B3139" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >De-multiplexing bronze table</text><text x="571" y="480" font-size="13.5" font-weight="400" fill="#1B3139" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >into intermediate tables</text><rect x="740" y="30" width="262" height="470" rx="2" fill="none" stroke="#8C98A0" stroke-width="1.6" stroke-dasharray="6 5" /><svg x="756" y="58" width="30" height="27.3" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="794" y="78" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="756" y="100" width="230" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="871.0" y="127" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >logistics_silver_demo</text><path d="M688,122 H752" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="756" y="188" width="30" height="27.3" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="794" y="208" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="756" y="230" width="230" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="871.0" y="257" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >marketing_silver_demo</text><path d="M688,252 H752" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="756" y="318" width="30" height="27.3" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="794" y="338" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="756" y="360" width="230" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="871.0" y="387" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >store_ops_silver_demo</text><path d="M688,382 H752" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><rect x="752" y="440" width="238" height="50" rx="0" fill="#F6F4F1" stroke="none" stroke-width="1" /><text x="871" y="470" font-size="14.5" text-anchor="middle" fill="#1B3139" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif">Performing <tspan font-weight="700">transformations</tspan></text><rect x="1040" y="30" width="250" height="340" rx="2" fill="none" stroke="#FFAB00" stroke-width="1.6" stroke-dasharray="6 5" /><svg x="1052" y="58" width="28" height="28" viewBox="1.265 25.6 47.748 47.748" fill="none"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#00ADD4"/></svg><text x="1088" y="78" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Delta Sink</text><rect x="1052" y="100" width="226" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="1165.0" y="127" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >delta_sink_logistics</text><path d="M988,122 H1048" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="1052" y="188" width="30" height="30" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><rect x="32.015" y="26.436" width="10.5" height="3.5" transform="rotate(131 37.265 28.186)"/><rect x="32.5" y="31" width="15" height="3"/><rect x="37.485" y="35.064" width="10.5" height="3.5" transform="rotate(131 42.735 36.814)"/></svg><text x="1090" y="208" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Materialized View</text><rect x="1052" y="230" width="226" height="44" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#om-sh)"/><text x="1165.0" y="257" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >marketing_campaign_summary</text><path d="M988,252 H1048" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#om-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><path d="M196,528 H980 V508 L1070,552 L980,596 V576 H196 z" fill="#1B5062"/><text x="560" y="559" font-size="16.5" text-anchor="middle" fill="#fff" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif"><tspan font-weight="700">Incrementally</tspan> process <tspan font-weight="700">new files</tspan> in the streaming tables (ST)</text></svg>

## A. Setup

> **Workspace setup:** Using a Databricks Academy (Vocareum) workspace? Your catalog is created for you, so you can start. Using your own workspace or Databricks Free Edition? Complete **Course Overview → B. Workspace Setup** first (catalog creation and permissions, Marketplace data). Run this notebook only in a development or sandbox workspace.

### A1. Configure Your Catalog and Schema

1. Run the cell below to initialize your environment. This setup step does the following:
    - **Assumes you have permission to create a catalog** when running outside of a Databricks provided Vocareum workspace
    - Create three schemas in your specified catalog:
        - **multiplex_1_bronze**
        - **multiplex_2_silver**
        - **multiplex_3_gold**
    - Create `business_events` volume in your **YOUR_LABUSER_CATALOG.multiplex_1_bronze** schema, and adds a single file to your volume.
    - Verifies your selected compute environment

    This ensures that all schemas, tables and objects are created in your catalog.

> **Important:** You must have permission to create catalogs in your own non Vocareum workspace. If you do not have the required permissions, this step will fail. If you cannot create catalogs, see **Course Overview → B2. Troubleshooting: Missing Create Catalog Permissions** before continuing.

In [ ]:
%run ./Includes/Classroom-Setup-multiplex

2. Run the cell below to view the value of the `my_vol_path` variable.

    Confirm that the value references your **your-catalog.multiplex_1_bronze** path. This will be used to dynamically reference your source volumes throughout this demonstration.

In [ ]:
print(my_vol_path)

## B. Explore the Raw Source Data for the Multiplex Data Ingestion

Before building the multiplex pipeline, start by exploring the raw source data stored in the volume.

The source volume receives multiplexed data from a Kafka stream, containing three distinct event groups:
- **store_ops**
- **marketing**
- **logistics**

All event types are ingested into a single file within a volume, simulating a real-world scenario where multiple business domains generate events concurrently.

For this demo, we process each file individually from the source location to demonstrate streaming ingestion and multiplexing.

### B1. Explore Business Events Daily Volume

1. View the files in the **multiplex_1_bronze.business_events** volume. 

    You will see one parquet file exists in this volume which contains event data.

In [ ]:
spark.sql(f"LIST '{my_vol_path}/business_events' ").display()

2. Query the raw Parquet in volume to view the raw data and note the following:
    - The data is packed in **BINARY** format
      - The **key** columns holds a unique identifier for the data 
      - The **volume** column holds the actual data.
      - The **topic** column indicates the different event types present in this source location

In [ ]:
%sql
SELECT * 
FROM read_files(
    my_vol_path || '/business_events'
)
LIMIT 10;

3. View the data by **casting `BINARY` columns to `STRING`** to view the values.

    Notice the following:
    - The **value_str** column contains a JSON string of key-value pairs, which vary depending on the business event group
    - The **topic** column indicates the different event types present in this source location


In [ ]:
%sql
SELECT 
  CAST(key AS STRING) AS key,
  CAST(value AS STRING) AS value_str,
  topic,
  partition,
  offset
FROM read_files(
    my_vol_path || '/business_events'
)
LIMIT 10;

### B2. Introduction to the Variant Data Type

1. The incoming event data is stored as JSON strings. Because each business event has a different structure, we use the **VARIANT** data type to handle this flexibility.

##### Why VARIANT:

- Supports semi structured JSON data
- Handles multiple event shapes in a single column
- Allows fields to be extracted when needed
- Works well for multiplex streaming data

##### What This Query Is Doing

This query converts raw JSON into a usable format.

- Reads raw event files with `read_files`
- Casts the raw value to a string
- Parses the JSON string into a VARIANT column
- Extracts `event_id` as a string
- Extracts and casts `timestamp` to a TIMESTAMP
- Keeps both raw and extracted fields for downstream use

VARIANT lets us ingest once, then shape the data later as part of the pipeline.

**NOTE:** This is a **quick introduction** to the VARIANT data type used in this notebook. We will use the `VARIANT` data type in your Spark Declarative Pipeline (SDP).

- [VARIANT type](https://docs.databricks.com/aws/en/sql/language-manual/data-types/variant-type)
- [Variant Data Type - Making Semi-Structured Data Fast and Simple - Deep Dive](https://www.youtube.com/watch?v=jtjOfggD4YY)
- [Introducing the Open Variant Data Type in Delta Lake and Apache Spark](https://www.databricks.com/blog/introducing-open-variant-data-type-delta-lake-and-apache-spark)

In [ ]:
%sql
SELECT 
  -- Raw JSON payload read from the file and cast to a string
  CAST(value AS STRING) AS value_str,

  -- Parse the JSON string into a VARIANT column for flexible field access
  parse_json(value_str) AS event_data_variant,

  -- Extract the event_id field from the VARIANT and cast it to STRING
  CAST(event_data_variant:event_id AS STRING) AS extracted_event_id,

  -- Extract the timestamp field from the VARIANT and cast it to TIMESTAMP
  event_data_variant:timestamp::TIMESTAMP AS extracted_timestamp
FROM read_files(
    my_vol_path || '/business_events'
)
LIMIT 10;

### B3. Analyze Raw Data Statistics

1. Explore the raw data for **business_events**. The cell below performs the following:
    - Counts the number of records in the file within the volume  
    - Counts the number of records in the file for each event group in our volume

    In the output, notice the following:
    - **372** rows are present in this file  
    - You will see counts for each of the three different sources

In [ ]:
# a. Total row count
df_count = spark.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM  read_files('{my_vol_path}/business_events')
""")
display(df_count)

# b. Data source count by topic
df_data_source_count = spark.sql(f"""
    SELECT 
        topic,
        COUNT(*) AS total_rows
    FROM read_files('{my_vol_path}/business_events')
    GROUP BY topic
    ORDER BY topic
""")
display(df_data_source_count)

## C. Create the Spark Declarative Pipeline

Now that we have explored the data and reviewed casting from `BINARY` to `STRING` along with the `VARIANT` data type, we are ready to create the Spark Declarative Pipeline.

### C1. Create a Apache Spark™ Declarative Pipeline Using the Lakeflow Pipelines Editor

Complete the following steps to create your Spark Declarative Pipeline:

1. In the main navigation pane, right-click **Jobs & Pipelines** and select **Open link in New Tab**.  

2. In the new tab, select **Create → ETL Pipeline**.  

3. At the top, complete the following:
   - Name your pipeline `demo_multiplex_yourname`
   - Select your default **catalog** and **schema**:  
        - **Catalog:** `YOUR_LABUSER_CATALOG`
        - **Schema:** **multiplex_1_bronze**  
      **NOTE:** Clear the selected schema using the cross icon to view all schemas.

4. Rename the **transformations** folder to `multiplex_pipeline`.

5. Rename the **my_transformation.py** file to `ingestion.sql`.

6. Leave the **Lakeflow Pipelines Editor** page open.

## D. Create the First Bronze Table for All Business Events Data

Start by creating the **bronze streaming table** to ingest data from the source.
- This table will capture raw business event data streaming in from multiple sources.
- We will use the `VARIANT` data type to parse JSON formatting strings.
- Two metadata columns are added to track when new data is ingested.

1. Copy the SQL code below and paste it into your `ingestion.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
------------------------------------------------------
-- CREATE THE BRONZE TABLE FOR BUSINESS EVENTS DATA
------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE multiplex_1_bronze.bronze_demo
TBLPROPERTIES (
  'pipelines.reset.allowed' = false
) 
AS
SELECT
  CAST(key AS STRING) AS event_id,
  PARSE_JSON(CAST(value AS STRING)) AS event_data_variant,
  CAST(topic AS STRING) AS event_group,
  CAST(partition AS STRING) AS partition,
  CAST(offset AS STRING) AS offset,
  -- Adding metadata columns
  _metadata.file_name AS source_file,
  _metadata.file_modification_time AS file_mod_time
FROM STREAM read_files(
  '${business_events_source}'
);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Code**

The `TBLPROPERTIES` statement configures the following setting:
- **Reset Protection**: The `'pipelines.reset.allowed' = false` property prevents full refreshes on the streaming table, which helps avoid accidentally removing checkpoints and truncating the streaming table data

#### IMPORTANT: Understanding Full Table Refresh Protection

This protection is particularly important when your raw data source automatically removes files after a certain timeframe. Without this setting, data that is no longer present in the source directory would not be re-ingested into the target table during a **Run pipeline with full table refresh** operation.

**NOTE:** For guidance on when to use full refreshes, see the [Should I use a full refresh?](https://docs.databricks.com/aws/en/ldp/updates#should-i-use-a-full-refresh) documentation.

## E. Fan Out the Bronze Table by Business Event

At this stage, we started with a single multiplexed bronze streaming table. The goal is to demultiplex this data by fanning it out into separate intermediate tables, one for each business event type.

This step creates three domain specific intermediate tables:

- **marketing_intermediate**
- **logistics_intermediate**
- **store_ops_intermediate**

Each table contains only the events relevant to its business domain, making downstream processing and analytics simpler and more focused.

### E1. Create the Marketing Bronze Table

Begin by creating **the** marketing intermediate table by:
- Filtering for records where `event_group = 'business_events_marketing'`
- Converting columns to their appropriate data types
- Including metadata columns to track ingested records


1. Copy the SQL code below and paste it into your `ingestion.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
------------------------------------------------------
-- CREATE MARKETING INTERMEDIATE TABLE
------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE multiplex_1_bronze.marketing_intermediate
AS SELECT 
  event_id,
  event_data_variant,
  event_group,
  event_data_variant:event_id::STRING AS extracted_event_id,
  event_data_variant:timestamp::TIMESTAMP AS timestamp,
  event_data_variant:event_type::STRING AS event_type,
  event_data_variant:subsidiary_id::STRING AS subsidiary_id,
  event_data_variant:campaign_id::STRING AS campaign_id,
  event_data_variant:channel::STRING AS channel,
  event_data_variant:impressions::LONG AS impressions,
  event_data_variant:clicks::LONG AS clicks,
  event_data_variant:conversions::LONG AS conversions,
  event_data_variant:spend_usd::DOUBLE AS spend_usd,
  source_file,
  file_mod_time
FROM STREAM multiplex_1_bronze.bronze_demo
WHERE event_group = 'business_events_marketing';
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### E2. Create the Logistics and Store Operations Bronze Tables

Now create the other two intermediate tables the same way you created the marketing table. Each one:
- Filters for its own events: `event_group = 'business_events_logistics'` for logistics and `event_group = 'business_events_store_ops'` for store operations
- Converts columns to their appropriate data types
- Includes metadata columns to track ingested records

**NOTE:** The store operations table names its parsed timestamp column `extracted_timestamp` (the marketing and logistics tables call it `timestamp`). The silver step renames it.

1. Copy the SQL code below and paste it into your `ingestion.sql` file. It creates **logistics_intermediate** and **store_ops_intermediate**.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
------------------------------------------------------
-- CREATE LOGISTICS INTERMEDIATE TABLE
------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE multiplex_1_bronze.logistics_intermediate
AS SELECT 
  event_id,
  event_data_variant,
  event_group,
  event_data_variant:event_id::STRING AS extracted_event_id,
  event_data_variant:timestamp::TIMESTAMP AS timestamp,
  event_data_variant:event_type::STRING AS event_type,
  event_data_variant:subsidiary_id::STRING AS subsidiary_id,
  event_data_variant:warehouse_id::STRING AS warehouse_id,
  event_data_variant:carrier::STRING AS carrier,
  event_data_variant:batch_id::STRING AS batch_id,
  event_data_variant:num_packages::LONG AS num_packages,
  event_data_variant:destination_region::STRING AS destination_region,
  source_file,
  file_mod_time
FROM STREAM multiplex_1_bronze.bronze_demo
WHERE event_group = 'business_events_logistics';

------------------------------------------------------
-- CREATE STORE OPERATIONS INTERMEDIATE TABLE
------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE multiplex_1_bronze.store_ops_intermediate
AS SELECT 
  event_id,
  event_data_variant,
  event_group,
  event_data_variant:event_id::STRING AS extracted_event_id,
  event_data_variant:timestamp::TIMESTAMP AS extracted_timestamp,
  event_data_variant:event_type::STRING AS event_type,
  event_data_variant:subsidiary_id::STRING AS subsidiary_id,
  event_data_variant:store_id::STRING AS store_id,
  event_data_variant:city::STRING AS city,
  event_data_variant:region::STRING AS region,
  event_data_variant:opened_by_employee_id::STRING AS opened_by_employee_id,
  source_file,
  file_mod_time
FROM STREAM multiplex_1_bronze.bronze_demo
WHERE event_group = 'business_events_store_ops';
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### E3. Configure the Pipeline Parameters

1. Run the cell below to retrieve the key-value pairs needed to set your pipeline configuration parameters for the **source volume**.

In [ ]:
config_parameters = [
    ('business_events_source', f'{my_vol_path}/business_events'),
    ('my_catalog',my_catalog)
]

for key, value in config_parameters:
    print(f"Key: {key}\nValue: {value}\n")

2. Copy the paths above and add each one as a configuration parameter in your **Spark Declarative Pipeline**.

    This will allow your pipeline to reference each volume through parameters.

   a. Select **Settings** in your pipeline tab  

   b. Under **Configuration**, select **Add configuration**

   c. For each **Key**, enter the key name shown above  

   d. For each **Value**, enter the corresponding volume path  

   e. Select **Save**

**NOTE:** For more details on configuration parameters, see the Databricks documentation: [Use parameters with Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/parameters)

### E4. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm that it completes successfully.

2. In the Lakeflow Pipelines editor, explore the pipeline run:
   - Verify that **372** rows were ingested into the bronze demo table from the `business_events` source volume
   - Confirm the row counts for each intermediate streaming table:
     - **194** rows in **logistics_intermediate**
     - **105** rows in **marketing_intermediate**
     - **73** rows in **store_ops_intermediate**
   - Select the **bronze_demo** table and open the **Data** tab to preview all of the ingested records business event data.

**TROUBLESHOOTING:** If your pipeline does not run successfully, make sure your volumes are created and your configuration parameters are set correctly.

#### Checkpoint

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.1.0/images/20261029T211417Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multiplex/checkpoint_bronze.png" alt="Bronze" width="1200">

## F. Create Silver Layer Tables

Next, you will create the silver layer tables that process and clean the bronze data for each business domain. Start by creating a new file in your **multiplex_pipeline**:

1. Click the kebab menu next to your **multiplex_pipeline** folder.

2. Select **Create file**

3. Select the language as **SQL**

4. Name the file `silver_transformation.sql`

### F1. Create the Silver Table for Marketing Data

In this step, we transform the raw marketing events into a curated silver table, **multiplex_2_silver.marketing_silver_demo**, which is optimized for analytics.

Here's what the silver transformation does:

- Reads data from the **marketing_intermediate** streaming table.
- Uses `COALESCE` to create a reliable **event_id**, ensuring each row has a unique identifier.
- Selects key business fields needed for analysis, such as campaign, channel, impressions, clicks, and spend.
- Keeps the data at the same level of detail (no aggregation or filtering that changes the number of rows).
- Calculates important marketing metrics:
  - **click_through_rate**: Measures how often people click after seeing an ad (clicks divided by impressions).
  - **cost_per_click**: Shows how much each click costs (spend divided by clicks).
- Safely handles cases where impressions or clicks are zero to prevent errors in metric calculations.
- Produces a clean, structured streaming table ready for dashboards and reporting.

This silver table ensures consistent data and trusted metrics for all downstream analytics.


1. Copy the code into your `silver_transformation.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
------------------------------------------------------
-- CREATE THE SILVER TABLE FOR MARKETING DATA
------------------------------------------------------

CREATE OR REFRESH STREAMING TABLE multiplex_2_silver.marketing_silver_demo
AS SELECT 
  COALESCE(extracted_event_id, event_id) AS event_id,
  timestamp,
  event_group,
  event_type,
  subsidiary_id,
  campaign_id,
  channel,
  impressions,
  clicks,
  conversions,
  spend_usd,
  CASE WHEN impressions > 0 
    THEN clicks / impressions 
    ELSE 0 
  END AS click_through_rate,
  CASE WHEN clicks > 0 
    THEN spend_usd / clicks 
    ELSE 0 
  END AS cost_per_click
FROM STREAM multiplex_1_bronze.marketing_intermediate;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### F2. Create the Silver Tables for Logistics and Store Operations Data

Now create the other two silver tables the same way as the marketing table. Each one reads from its intermediate streaming table, uses `COALESCE` to build a reliable **event_id**, applies data quality filters and adds calculated columns.

**multiplex_2_silver.logistics_silver_demo** (reads **logistics_intermediate**)
- Filters out incomplete records: rows where **warehouse_id** or **batch_id** are missing.
- Selects the warehouse, carrier, batch ID, package count and destination region.
- Adds **is_valid_shipment** (true if `num_packages` > 0, which helps identify ghost shipments or system errors) and **event_date** (the date from the timestamp).

**multiplex_2_silver.store_ops_silver_demo** (reads **store_ops_intermediate**)
- Filters out records that are missing the timestamp, store ID or event type.
- Renames **extracted_timestamp** to the common **timestamp** column.
- Adds **event_date**, **event_hour** and **store_number** (parsed from **store_id** with a split function).

1. Copy the code into your `silver_transformation.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
------------------------------------------------------
-- CREATE THE SILVER TABLE FOR LOGISTICS DATA
------------------------------------------------------

CREATE OR REFRESH STREAMING TABLE multiplex_2_silver.logistics_silver_demo
AS SELECT 
  COALESCE(extracted_event_id, event_id) AS event_id,
  timestamp,
  event_group,
  event_type,
  subsidiary_id,
  warehouse_id,
  carrier,
  batch_id,
  num_packages,
  destination_region,
  CASE WHEN num_packages > 0 THEN TRUE ELSE FALSE END AS is_valid_shipment,
  DATE(timestamp) AS event_date
FROM STREAM multiplex_1_bronze.logistics_intermediate
WHERE warehouse_id IS NOT NULL
  AND batch_id IS NOT NULL;

------------------------------------------------------
-- CREATE THE SILVER TABLE FOR STORE OPERATIONS DATA
------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE multiplex_2_silver.store_ops_silver_demo
AS SELECT 
  COALESCE(extracted_event_id, event_id) AS event_id,
  extracted_timestamp AS timestamp,
  event_group,
  event_type,
  subsidiary_id,
  store_id,
  city,
  region,
  opened_by_employee_id,
  DATE(extracted_timestamp) AS event_date,
  HOUR(extracted_timestamp) AS event_hour,
  SPLIT(store_id, '_')[2] AS store_number
FROM STREAM multiplex_1_bronze.store_ops_intermediate
WHERE extracted_timestamp IS NOT NULL
  AND store_id IS NOT NULL
  AND event_type IS NOT NULL;
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### F3. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm it runs successfully 

2. In the Lakeflow Pipelines editor, explore the pipeline run:
   - Confirm the row counts for each silver streaming table:
     - **194** rows in **logistics_silver_demo**
     - **105** rows in **marketing_silver_demo**
     - **73** rows in **store_ops_silver_demo**
   - Select any silver table and open the **Data** tab to preview the results

**TROUBLESHOOTING:** If your pipeline does not run successfully, confirm that your volumes were created and that your configuration parameters are set correctly.

#### Checkpoint

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.1.0/images/20261029T211417Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multiplex/checkpoint_silver.png" alt="silver" width="1200">

### F4. Query the Silver Streaming Tables

1. Verify that the data has been properly demultiplexed into the silver layer tables.

In [ ]:
%sql
SELECT * 
FROM multiplex_2_silver.marketing_silver_demo

In [ ]:
%sql
SELECT * 
FROM multiplex_2_silver.logistics_silver_demo

In [ ]:
%sql
SELECT * 
FROM multiplex_2_silver.store_ops_silver_demo

## G. Create Gold Layer Views

With the silver tables in place, we can now build the gold layer.

The gold layer consists of materialized views that:

- Aggregate data from the silver tables
- Apply business friendly calculations and groupings
- Provide stable, analytics ready datasets for dashboards and reporting
- Reduce query complexity for downstream consumers

In this section, we will create a materialized view on top of the marketing silver table to support common business questions.

**NOTE:** For this training, we will create a single materialized view. In a real production scenario, you would typically create multiple materialized views to support different analytics needs.

### G1. Create a New SQL File in Your Pipeline

1. On your **multiplex_pipeline** folder select the kebab menu and select **Create File**

2. Select the language as **SQL**

3. Name the file `gold_view.sql`

### G2. Create Gold View for Marketing Campaign Summary
In this step, we elevate the data to the Gold layer by creating a materialized view, **multiplex_3_gold.marketing_campaign_summary**. This view aggregates the granular event data into high-level performance metrics, optimized for executive reporting and dashboards.

Here's what this transformation does:

- Aggregates data from the **marketing_silver_demo** table, grouping it by Campaign, Subsidiary, and Channel to provide a clear performance summary.
- **Calculates Totals:** Sums up key volume metrics (events, impressions, clicks, conversions) and financial metrics (total spend) for each group.
- Derives Strategic KPIs:
  - **ctr_percentage**: The Click-Through Rate formatted as a percentage.
  - **conversion_rate_percentage**: The percentage of clicks that resulted in a successful conversion.
  - **cost_per_conversion**: The average amount spent to acquire a single conversion (CPA).
- Ensures Data Quality:
  - Uses `NULLIF` to prevent "division by zero" errors when calculating ratios.
  - Uses `ROUND` to format currency and percentages to two decimal places for cleaner presentation.

This materialized view serves as the "source of truth" for campaign performance dashboards, enabling stakeholders to compare channel effectiveness and ROI instantly.

1. Copy the code below in your `gold_view.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
------------------------------------------------------
-- CREATE MATERIALIZED VIEW FOR MARKETING DATA
------------------------------------------------------

CREATE OR REFRESH MATERIALIZED VIEW multiplex_3_gold.marketing_campaign_summary
AS SELECT
  campaign_id,
  subsidiary_id,
  channel,
  COUNT(*) AS total_events,
  SUM(impressions) AS total_impressions,
  SUM(clicks) AS total_clicks,
  SUM(conversions) AS total_conversions,
  ROUND(SUM(spend_usd), 2) AS total_spend_usd,
  ROUND((SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0)) * 100, 2) AS ctr_percentage,
  ROUND((SUM(conversions) * 1.0 / NULLIF(SUM(clicks), 0)) * 100, 2) AS conversion_rate_percentage,
  ROUND(SUM(spend_usd) / NULLIF(SUM(conversions), 0), 2) AS cost_per_conversion
FROM multiplex_2_silver.marketing_silver_demo
GROUP BY campaign_id, subsidiary_id, channel;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### G3. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm it runs successfully 

2. Explore the run in the Lakeflow Pipelines Editor:
   - Confirm **96** rows were ingested into the **marketing_campaign_summary** materialized view
   - Preview the data in the editor (Select **marketing_campaign_summary** → **Data** tab)

**TROUBLESHOOTING:** If your pipeline does not run successfully, confirm that your volumes were created and that your configuration parameters are set correctly.

In [ ]:
%sql
SELECT * 
FROM multiplex_3_gold.marketing_campaign_summary

## H. Enable Iceberg Reads

Now, we want to enable Iceberg reads on our final streaming table to support cross-platform analytics and data sharing.

### H1. Create a Delta Sink

We need to create a Delta sink table because Iceberg reads (UniForm with Iceberg v2) cannot be enabled directly on streaming tables or views. To do this, we will create a Delta sink from one of our streaming tables.

1. What are sinks in Spark Declarative Pipelines, and how do we use them?

- [Sinks in Apache Spark™ Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/sinks)

- [Using sinks in pipelines](https://docs.databricks.com/aws/en/ldp/ldp-sinks)

2. Begin by creating a new Python File in Your Pipeline to create the **delta_sink_logistics** sink . 
    - **NOTE: Only the Python API is supported. SQL is not supported.**

   a. Click the kebab menu on your **multiplex_pipeline** folder and select **Create file**

   b. Select the language as **Python**

   c. Name the file `delta_sink.py`

   d. Copy the code below and paste in your `delta_sink.py` file.

3. Here is what the code below does:
- **Defines a Destination (Sink):** The `create_sink` function registers a specific endpoint for your data—a Delta table named `logistics_delta_sink` in the Gold schema—separately from the logic that populates it.
- **Establishes a Streaming Flow:** The `@dp.append_flow` decorator creates a continuous data pipeline that links your processing logic directly to the defined sink, automating the movement of data.
- **Enables Append-Only Logic:** The append flow optimizes performance by adding new records as they arrive, rather than re-processing or overwriting existing data.
- **Automates Incremental Processing:** By using `readStream`, the pipeline automatically tracks which data has already been processed (using checkpointing), ensuring that only new records from the Silver table are moved to Gold.

4. Copy and paste into your `delta_sink.py` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
# ------------------------------------------------------
#       CREATE DELTA SINK
# ------------------------------------------------------

from pyspark import pipelines as dp

my_catalog = spark.conf.get("my_catalog")

dp.create_sink(
  name = "delta_sink_logistics",
  format = "delta",
  options = { "tableName": f"{my_catalog}.multiplex_3_gold.logistics_delta_sink" }
)

@dp.append_flow(name = "delta_sink_logistics_flow", target="delta_sink_logistics")
def delta_sink_logistics_flow():
  return(
  spark.readStream.table("multiplex_2_silver.logistics_silver_demo")
)

</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### H2. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm it runs successfully 

2. Review the pipeline results in the Lakeflow Pipelines Editor:
   - Ensure the **marketing_campaign_summary** materialized view contains **96** records.
   - Check that the record count in **logistics_silver_demo** matches the output in **delta_sink_logistics** (should be **194** records).

---

#### Pipeline Final State

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.1.0/images/20261029T211417Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multiplex/checkpoint_final.png" alt="final" width="1200">

### H3. Examine Table Properties

1. Review the table properties and note the following:
- In **Detailed Table Information**, confirm that the **type** is **managed** and the **provider** is **delta**. This indicates it is a managed Delta table.
- In the **table_properties** column of the same section, you will see that deletion vectors are enabled: `delta.enableDeletionVectors=true`.


In [ ]:
%sql
DESCRIBE EXTENDED multiplex_3_gold.logistics_delta_sink

### H4. Enable Iceberg Reads

1. To enable Iceberg reads on a Delta table, you must **disable deletion vectors**. Deletion vectors allow soft deletes, but Iceberg requires hard deletes for compatibility.

      **How to enable Iceberg reads:**

      a. **Disable deletion vectors:** Turn off deletion vectors for your Delta table.


      b. **Enable Iceberg compatibility:** Set the table property to allow Iceberg reads.

> **NOTE:**  
> With [Iceberg v3](https://docs.databricks.com/aws/en/iceberg/iceberg-v3) (generally available since May 21, 2026; requires Databricks Runtime 18 LTS or above), you do **not** need to disable deletion vectors. 

For more details, see the [Databricks documentation on deletion vectors](https://docs.databricks.com/aws/en/delta/deletion-vectors).

In [ ]:
%sql
-- Step 1: Disable deletion vectors
ALTER TABLE multiplex_3_gold.logistics_delta_sink 
  SET TBLPROPERTIES (
    'delta.enableDeletionVectors' = 'false'
  );

-- Step 2: Enable Iceberg compatibility
ALTER TABLE multiplex_3_gold.logistics_delta_sink 
  SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.enableIcebergCompatV2' = 'true',
    'delta.universalFormat.enabledFormats' = 'iceberg'
  );

2. Let's review the table properties and observe the changes:
- You will see a new section called **Delta Uniform Iceberg**, which includes details such as 
  - **metadata location**, 
  - **converted delta version**,
  - **converted delta timestamp**

In [ ]:
%sql
DESCRIBE TABLE EXTENDED multiplex_3_gold.logistics_delta_sink;

3. Let's view the table properties:
- `delta.universalFormat.enabledFormats = iceberg` confirms that Iceberg format is enabled.
- `delta.enableDeletionVectors = false` shows that deletion vectors is not enabled.
- The table currently supports Iceberg version 2: `delta.feature.icebergCompatV2 = supported`.

In [ ]:
%sql
SHOW TBLPROPERTIES multiplex_3_gold.logistics_delta_sink;

## I. Land Another File in Cloud Storage

Earlier in the workshop, we ingested a single file. In this step, we will land an additional file into the cloud storage volume to demonstrate how Spark Declarative Pipelines respond when new data is detected.

1. Run the command below to add the new file to your source volume, then confirm that the volume now contains two files.


In [ ]:
ops_path = f'/Volumes/{my_catalog}/multiplex_1_bronze/ops'
business_events_source_path = f'/Volumes/{my_catalog}/multiplex_1_bronze/business_events'


#copy files from Ops location to user's source volume
copy_files(copy_from = f'{ops_path}', copy_to = business_events_source_path, n = 2)

spark.sql(f"LIST '{business_events_source_path}' ").display()

2. Let's review the number of rows present in the newly landed file that will be ingested by our pipeline:
- **Marketing**: 62 new records
- **Store Ops**: 80 new records
- **Logistics**: 158 new records
- **Total**: 300 new records in this file

In [ ]:
%sql
SELECT 
    topic,
    count(*) as number_of_records_per_event_group , 
    sum(count(*)) OVER () as total_records_in_raw_file
FROM read_files(
    my_vol_path || '/business_events/part-02_business_events.parquet'
)
GROUP BY topic;

3. Run the Spark Declarative Pipeline to incrementally process the new file and confirm that it completes successfully.

4. In the Lakeflow Pipelines editor, review the pipeline run:
   - Verify that **300** rows were ingested into the bronze demo table from the `business_events` source volume with the new landed file.
   - Confirm the row counts for each intermediate bronze and silver streaming table:
     - **158** rows in both **logistics_intermediate** and **logistics_silver_demo**
     - **80** rows in both **store_ops_intermediate** and **store_ops_silver_demo**
     - **62** rows in both **marketing_intermediate** and **marketing_silver_demo**
   - Verify that the materialized view **marketing_campaign_summary** contains **141** output records.
   - Ensure that **delta_sink_logistics** has the same number of records as **logistics_silver_demo** - **158** records.


## J. Summary and Key Takeaways 
- **Multiplex Data Pipeline Architecture**: Successfully built a complete multiplex streaming pipeline using Spark Declarative Pipelines that ingested data from a single source and demultiplexed it into three separate business domain tables (marketing, logistics, and store operations)
- **VARIANT Data Type Implementation**: Leveraged the VARIANT data type to efficiently handle semi-structured JSON data with different schemas across multiple event types in a single bronze table
- **Medallion Architecture with Streaming**: Implemented the full medallion pattern (bronze → silver → gold) using streaming tables, intermediate transformations, and materialized views for real-time data processing
- **Delta Sinks and Iceberg Compatibility**: Created Delta sinks using the Python API and enabled Iceberg compatibility for cross-platform analytics by disabling deletion vectors and configuring universal format properties

#### Business Value Delivered

The pipeline processed **672 total records** from two source files containing multiplexed business events, creating domain-specific analytics tables that enable:
- Real-time operational insights across marketing campaigns, logistics shipments, and store operations
- Cross-platform data access through Iceberg compatibility for broader ecosystem integration  
- Scalable event processing architecture that can handle additional business domains and event types
- Automated incremental processing with streaming checkpoints ensuring exactly-once delivery

This multiplex architecture provides a foundation for enterprise-scale event processing while maintaining data lineage, real-time capabilities, and cross-platform compatibility through Delta Universal Format.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>